# Módulo 1 — LSTM: Predicción de Demanda de Transporte
**Datos reales**: `amanmehra23/travel-recommendation-dataset` (Kaggle)

Universidad Nacional de Colombia · IRNA 2026-01

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, pickle, sys
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cpu')

LOOKBACK=30; BATCH_SIZE=32; EPOCHS=150; LR=5e-4; PATIENCE=15; MIN_DAYS=60
BASE_DIR = Path('.'); MODELS_DIR = BASE_DIR/'models'; MODELS_DIR.mkdir(exist_ok=True)
print("Config: LOOKBACK=%d | EPOCHS=%d | LR=%s | device=%s" % (LOOKBACK,EPOCHS,LR,device))

## 1. Carga del dataset real de Kaggle

In [ ]:
import kagglehub
print("Descargando dataset...")
try:
    raw_path = Path(kagglehub.dataset_download("amanmehra23/travel-recommendation-dataset"))
except Exception as e:
    sys.exit(f"ERROR: {e}\nConfigura ~/.kaggle/kaggle.json")

csvs = {p.stem.lower(): p for p in raw_path.rglob("*.csv")}
reviews_path = next((p for k,p in csvs.items() if "review" in k), None)
dest_path    = next((p for k,p in csvs.items() if "destination" in k), None)
assert reviews_path and dest_path, f"No encontré CSVs. Disponibles: {list(csvs.values())}"

df_reviews = pd.read_csv(reviews_path)
df_dest    = pd.read_csv(dest_path)
print(f"Reviews: {df_reviews.shape} | Destinations: {df_dest.shape}")

## 2. Construcción de series temporales reales

In [ ]:
# Detectar columna de fecha
date_col = None
for col in df_reviews.columns:
    try:
        pd.to_datetime(df_reviews[col].dropna().head(10), errors='raise')
        date_col = col; break
    except: pass
assert date_col, f"No se encontró columna de fecha. Columnas: {list(df_reviews.columns)}"

df_reviews[date_col] = pd.to_datetime(df_reviews[date_col], errors='coerce')
df_reviews = df_reviews.dropna(subset=[date_col])

# Join para obtener nombre del destino
if "DestinationID" in df_reviews.columns and "Name" in df_dest.columns:
    df_reviews = df_reviews.merge(
        df_dest[["DestinationID","Name"]].drop_duplicates("DestinationID"),
        on="DestinationID", how="left")
    dest_col = "Name"
else:
    dest_col = next(c for c in df_reviews.columns if df_reviews[c].dtype==object)

df_reviews["date_only"] = df_reviews[date_col].dt.normalize()
daily = df_reviews.groupby([dest_col,"date_only"]).size().reset_index(name="demand")

dest_days = daily.groupby(dest_col)["date_only"].nunique()
top5 = dest_days[dest_days>=MIN_DAYS].sort_values(ascending=False).head(5).index.tolist()
assert top5, f"Ningún destino tiene ≥{MIN_DAYS} días de datos."
print(f"Top 5 destinos seleccionados: {top5}")

# Construir series con relleno
series_dict = {}
for dest in top5:
    sub = daily[daily[dest_col]==dest].set_index("date_only")["demand"].sort_index()
    full_idx = pd.date_range(sub.index.min(), sub.index.max(), freq="D")
    sub = sub.reindex(full_idx).interpolate("linear").ffill().bfill().clip(lower=0)
    series_dict[dest] = sub
    print(f"  {dest}: {len(sub)} días | media={sub.mean():.1f} | max={sub.max():.0f}")

## 3. Arquitectura LSTM

In [ ]:
class LSTMDemanda(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=3, dropout=0.25):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers>1 else 0.0)
        self.head = nn.Sequential(
            nn.Linear(hidden_size,64), nn.ReLU(), nn.Dropout(0.1), nn.Linear(64,1))
    def forward(self,x):
        out,_ = self.lstm(x); return self.head(out[:,-1,:])

class DemandDataset(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X).unsqueeze(-1)
        self.y=torch.tensor(y).unsqueeze(-1)
    def __len__(self): return len(self.X)
    def __getitem__(self,i): return self.X[i],self.y[i]

def create_sequences(vals, lookback):
    X,y=[],[]
    for i in range(len(vals)-lookback): X.append(vals[i:i+lookback]); y.append(vals[i+lookback])
    return np.array(X,dtype=np.float32),np.array(y,dtype=np.float32)

print("Arquitectura LSTM: input=1 | hidden=128 | layers=3 | dropout=0.25")
# Test rápido
m=LSTMDemanda(); x=torch.zeros(4,LOOKBACK,1); print("Output shape:",m(x).shape)

## 4. Entrenamiento por destino

In [ ]:
def mape(y_true,y_pred,eps=1e-6):
    return np.mean(np.abs((y_true-y_pred)/(y_true+eps)))*100.0

models_state={}; scalers={}; routes_metadata={}; all_metrics=[]

for dest, series in series_dict.items():
    print(f"\nEntrenando → {dest} ({len(series)} días)")
    scaler = MinMaxScaler(feature_range=(0,1))
    vals_sc = scaler.fit_transform(series.values.reshape(-1,1)).flatten()
    X,y = create_sequences(vals_sc, LOOKBACK)
    n=len(X); tr_end=int(n*0.75); va_end=int(n*0.88)
    tr_ld=DataLoader(DemandDataset(X[:tr_end],y[:tr_end]),batch_size=BATCH_SIZE,shuffle=True)
    va_ld=DataLoader(DemandDataset(X[tr_end:va_end],y[tr_end:va_end]),batch_size=BATCH_SIZE)
    
    model=LSTMDemanda().to(device)
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=1e-5)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    crit=nn.HuberLoss(delta=1.0)
    best_val=float('inf'); best_st=None; pat=0; hist_va=[]
    
    for ep in range(1,EPOCHS+1):
        model.train()
        for xb,yb in tr_ld:
            xb,yb=xb.to(device),yb.to(device); opt.zero_grad()
            loss=crit(model(xb),yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        sch.step()
        model.eval(); vl=0.0
        with torch.no_grad():
            for xb,yb in va_ld:
                vl+=crit(model(xb.to(device)),yb.to(device)).item()*len(xb)
        vl/=max(len(va_ld.dataset),1); hist_va.append(vl)
        if vl<best_val: best_val=vl; best_st={k:v.clone() for k,v in model.state_dict().items()}; pat=0
        else:
            pat+=1
            if pat>=PATIENCE: print(f"  Early stop epoch {ep}"); break
    
    model.load_state_dict(best_st)
    Xte=torch.tensor(X[va_end:]).unsqueeze(-1).to(device)
    with torch.no_grad(): ps=model(Xte).cpu().numpy().flatten()
    acts=scaler.inverse_transform(y[va_end:].reshape(-1,1)).flatten()
    preds=np.clip(scaler.inverse_transform(ps.reshape(-1,1)).flatten(),0,None)
    rmse=float(np.sqrt(mean_squared_error(acts,preds)))
    mae=float(mean_absolute_error(acts,preds))
    mape_=float(mape(acts,preds))
    print(f"  RMSE={rmse:.2f} | MAE={mae:.2f} | MAPE={mape_:.2f}%")
    
    models_state[dest]=model.state_dict(); scalers[dest]=scaler
    vals_sc_all=scaler.transform(series.values.reshape(-1,1)).flatten()
    # Pronóstico 30 días
    window=list(vals_sc_all[-LOOKBACK:])
    fore_sc=[]
    model.eval()
    with torch.no_grad():
        for _ in range(30):
            x=torch.tensor(window[-LOOKBACK:],dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
            p=float(model(x).item()); p=max(0.,min(1.,p)); fore_sc.append(p); window.append(p)
    forecast=np.clip(scaler.inverse_transform(np.array(fore_sc).reshape(-1,1)).flatten(),0,None)
    
    routes_metadata[dest]={
        'n_days':len(series),'date_start':str(series.index[0].date()),
        'date_end':str(series.index[-1].date()),'lookback':LOOKBACK,
        'demand_mean':float(series.mean()),'demand_max':float(series.max()),
        'metrics':{'RMSE':rmse,'MAE':mae,'MAPE (%)':mape_},
        'forecast_30d':forecast.tolist(),
        'last_30d':series.values[-30:].tolist(),
        'last_30d_dates':[str(d.date()) for d in series.index[-30:]]
    }
    all_metrics.append({'Destino':dest,'RMSE':rmse,'MAE':mae,'MAPE (%)':mape_})

print("\n=== MÉTRICAS FINALES ===")
print(pd.DataFrame(all_metrics).to_string(index=False))

## 5. Visualización: predicción vs real

In [ ]:
fig, axes = plt.subplots(len(series_dict),1,figsize=(14,4*len(series_dict)))
if len(series_dict)==1: axes=[axes]
for ax, (dest, series) in zip(axes, series_dict.items()):
    scaler=scalers[dest]
    vals_sc=scaler.transform(series.values.reshape(-1,1)).flatten()
    X,y=create_sequences(vals_sc,LOOKBACK)
    n=len(X); va_end=int(n*0.88)
    model=LSTMDemanda(); model.load_state_dict(models_state[dest]); model.eval()
    Xte=torch.tensor(X[va_end:]).unsqueeze(-1)
    with torch.no_grad(): ps=model(Xte).numpy().flatten()
    acts=scaler.inverse_transform(y[va_end:].reshape(-1,1)).flatten()
    preds=np.clip(scaler.inverse_transform(ps.reshape(-1,1)).flatten(),0,None)
    test_idx=series.index[va_end+LOOKBACK:][:len(preds)]
    ax.plot(test_idx, acts[:len(test_idx)], label='Real',color='#2C5282',linewidth=1.5)
    ax.plot(test_idx, preds[:len(test_idx)],label='Predicción',color='#E53E3E',linewidth=1.5,linestyle='--')
    m=routes_metadata[dest]['metrics']
    ax.set_title(f"{dest} | RMSE={m['RMSE']:.2f} MAE={m['MAE']:.2f} MAPE={m['MAPE (%)']:.2f}%")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_pred_vs_real.png',dpi=120,bbox_inches='tight'); plt.show()
print("Guardado: fig_pred_vs_real.png")

## 6. Pronóstico 30 días

In [ ]:
fig,axes=plt.subplots(len(series_dict),1,figsize=(14,4*len(series_dict)))
if len(series_dict)==1: axes=[axes]
for ax,(dest,series) in zip(axes,series_dict.items()):
    meta=routes_metadata[dest]
    last_dates=[pd.Timestamp(d) for d in meta['last_30d_dates']]
    last_vals=meta['last_30d']
    last_dt=pd.Timestamp(meta['last_30d_dates'][-1])
    fore_dates=pd.date_range(last_dt+pd.Timedelta(days=1),periods=30,freq='D')
    fore_vals=meta['forecast_30d']
    ax.plot(last_dates,last_vals,color='#2C5282',label='Histórico real',linewidth=2)
    ax.plot(fore_dates,fore_vals,color='#E53E3E',label='Pronóstico 30d',linewidth=2,linestyle='--')
    ax.fill_between(fore_dates,
        np.array(fore_vals)*0.85, np.array(fore_vals)*1.15,
        alpha=0.2, color='#E53E3E', label='Banda ±15%')
    ax.axvline(x=last_dt,color='gray',linestyle=':',linewidth=1)
    ax.set_title(f"Pronóstico 30 días — {dest}"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_forecast_30d.png',dpi=120,bbox_inches='tight'); plt.show()

## 7. Guardar artefactos

In [ ]:
torch.save(models_state, MODELS_DIR/'lstm_demanda.pt')
with open(MODELS_DIR/'scaler_demanda.pkl','wb') as f: pickle.dump(scalers,f)
with open(MODELS_DIR/'routes_metadata.pkl','wb') as f: pickle.dump(routes_metadata,f)
# Copiar también a webapp/models
import shutil
webapp_models = Path('../webapp/models'); webapp_models.mkdir(exist_ok=True)
for fname in ['lstm_demanda.pt','scaler_demanda.pkl','routes_metadata.pkl']:
    shutil.copy(MODELS_DIR/fname, webapp_models/fname)
print("[OK] Artefactos guardados en models/ y ../webapp/models/")
for dest,meta in routes_metadata.items():
    m=meta['metrics']
    print(f"  {dest}: RMSE={m['RMSE']:.2f} | MAE={m['MAE']:.2f} | MAPE={m['MAPE (%)']:.2f}%")